# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Business Transformation and Modeling

%md
## Bảng Chiều Người Vay (dim_borrower)

In [0]:
query_dim_borrower = """
WITH distinct_borrowers AS (
    SELECT DISTINCT
        member_id,
        employment_title,
        employment_length_years,
        home_ownership,
        annual_income,
        address_state,
        dti,
        has_prior_delinq,
        has_major_derog
    FROM silver.lc_loans
    WHERE member_id IS NOT NULL
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY member_id) AS borrower_key,
    *
FROM distinct_borrowers
"""

df_dim_borrower = spark.sql(query_dim_borrower)
(
    df_dim_borrower.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_borrower")
)

## Bảng Chiều Khoản Vay (dim_loan_info)

In [0]:
query_dim_loan_info = """
WITH distinct_loans AS (
    SELECT DISTINCT
        id AS loan_id,
        grade,
        sub_grade,
        loan_term_months,
        interest_rate,
        loan_purpose,
        verification_status
    FROM silver.lc_loans
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY loan_id) AS loan_info_key,
    *
FROM distinct_loans
"""

df_dim_loan_info = spark.sql(query_dim_loan_info)

(
    df_dim_loan_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_loan_info")
)

## Bảng Sự Kiện Trung Tâm (fact_loans)

In [0]:
query_fact_loans = """
SELECT 
    ROW_NUMBER() OVER (ORDER BY id) AS fact_key,
    id AS loan_id,
    member_id,
    issue_date AS disbursement_date,
    loan_status,
    loan_amount,
    funded_amount,
    installment,
    total_payment_received,
    outstanding_principal,
    total_principal_received,
    total_interest_received,
    total_late_fees_received,
    recoveries
FROM silver.lc_loans
"""

df_fact_loans = spark.sql(query_fact_loans)

(
    df_fact_loans.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_loans")
)